# **Regression - Yes Bank Stock Closing Price Prediction**    -



##### **Project Type**    - Regression
##### **Contribution**    - Individual
##### **Name**    - Chaitanya Kiran Jaunjal

# **Project Summary -**

This project focuses on predicting the monthly closing stock price of Yes Bank using machine learning techniques. The dataset consists of monthly stock prices, including opening, highest, lowest, and closing prices recorded since the bank's inception.

The project began with data loading, cleaning, and preprocessing to ensure data quality and consistency. Exploratory Data Analysis (EDA) was then performed using statistical summaries and visualizations to understand the distribution of variables, identify trends, detect outliers, and analyze relationships among stock price attributes.

Feature engineering and data transformation techniques were applied where necessary to improve model performance. Multiple regression models were developed and evaluated using appropriate performance metrics such as R² Score, Mean Absolute Error (MAE), Mean Squared Error (MSE), and Root Mean Squared Error (RMSE).

To enhance transparency and interpretability, SHAP (SHapley Additive exPlanations) was used to analyze feature importance and understand how different stock price variables contributed to model predictions. The final model was selected based on its predictive performance and ability to generalize to unseen data.

The project demonstrates how machine learning techniques can be utilized to analyze historical stock market data and generate meaningful predictions for financial decision-making.


# **GitHub Link -**

https://github.com/jchaitanyak07/Regression---Yes-Bank-Stock-Closing-Price-Prediction

# **Problem Statement**


**The objective of this project is to predict the monthly closing stock price of Yes Bank using historical stock market data. The dataset contains the opening, highest, lowest, and closing prices of the stock for each month since the bank's inception.

Yes Bank has been one of India's prominent private sector banks. However, the bank faced significant challenges after the fraud case involving its founder, Rana Kapoor, which attracted widespread attention and impacted investor confidence. These events resulted in considerable fluctuations in the bank's stock prices.

The goal of this project is to analyze historical stock price data, identify patterns and relationships among stock variables, and build machine learning models capable of accurately predicting the monthly closing price. The project also aims to evaluate the effectiveness of different regression algorithms and understand the factors influencing stock price movements through model explainability techniques.
**

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import shap

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

### Dataset Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/data_YesBank_StockPrices.csv")

### Dataset First View

In [ ]:
# Here first five rows will be shown
data.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
data.shape

### Dataset Information

In [ ]:
# Dataset Info
data.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
data.duplicated().sum()

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
data.isna().sum()

In [ ]:
data.isnull().sum()

####Visualizing the missing values per column

In [ ]:
# Count null values
miss_counts_df = data.isna().sum().reset_index()
miss_counts_df = miss_counts_df.rename(columns = {"index" : "Columns", 0 : "Counts"})

# Plot
sns.barplot(data = miss_counts_df, x = "Columns", y = "Counts")
plt.title("Missing Values per Column")
plt.show()

####Visualizing Percentage of missing values per column

In [ ]:
# Count missing values percentage in each column
miss_counts_df_2 = ((data.isna().sum() / data.shape[0]) * 100).reset_index()
miss_counts_df_2 = miss_counts_df_2.rename(columns = {"index" : "Columns", 0 : "Percentage"})

# Plot
sns.barplot(data = miss_counts_df_2, x = "Columns", y = "Percentage")
plt.show()

### What did you know about your dataset?

* There are total 185 rows and 5 columns in the dataset.

* The dataset doesn't contain any missing or null or duplicate values.

* Columns 'Date' is an 'object' datatype and rest all columns are of 'float' datatype.

* Column 'Close' is dependent column (target variable) and rest other are independent columns.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
data.columns

In [ ]:
# Dataset Describe
data.describe()

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
cols = list(data.columns)
col_unique_dict = {}

for col in cols:
  col_unique_dict[col] = data[col].unique()

col_unique_dict

## 3. ***Data Wrangling***

### Data Wrangling Code

####Separate Month from Date Column

In [ ]:
data["Month"] = data["Date"].apply(lambda date_row : date_row.split("-")[0])

In [ ]:
data.columns

In [ ]:
data["Month"].head()

In [ ]:
data["Month"].shape

In [ ]:
data["Month"].unique()

####Cyclical Encoding on Month Column

In [ ]:
month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

In [ ]:
data["Month_Num"] = data["Month"].map(month_map)

In [ ]:
data["Month_Num"].shape

In [ ]:
data["Month_Num"].head()

In [ ]:
data["Month_Sin"] = np.sin((2 * np.pi * data["Month_Num"]) / 12)
data["Month_Cos"] = np.cos((2 * np.pi * data["Month_Num"]) / 12)

In [ ]:
data.head()

####'Low' < 'Open' < 'High'

In [ ]:
data[(data["Open"] > data["High"]) | (data["Open"] < data["Low"])]

####Column 'Open'

In [ ]:
data["Open"].describe()

In [ ]:
sns.boxplot(data = data, x = "Open")
plt.show()

In [ ]:
open_arr = np.array(data["Open"])

In [ ]:
Q1 = np.percentile(open_arr, 25)
Q3 = np.percentile(open_arr, 75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1: ", Q1)
print("Q3: ", Q3)
print("IQR: ", IQR)
print()
print("Lower bound: ", lower_bound)
print("Upper bound: ", upper_bound)

In [ ]:
data[(data["Open"] > upper_bound) | (data["Open"] < lower_bound)].shape

In [ ]:
data = data[(data["Open"] <= upper_bound) & (data["Open"] >= lower_bound)]

In [ ]:
data.shape

####Column 'High'

In [ ]:
data["High"].describe()

In [ ]:
sns.boxplot(data = data, x = "High")
plt.show()

In [ ]:
high_arr = np.array(data["High"])

In [ ]:
Q1 = np.percentile(high_arr, 25)
Q3 = np.percentile(high_arr, 75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1: ", Q1)
print("Q3: ", Q3)
print("IQR: ", IQR)
print()
print("Lower bound: ", lower_bound)
print("Upper bound: ", upper_bound)

In [ ]:
data[(data["High"] > upper_bound) | (data["High"] < lower_bound)]

In [ ]:
data = data[(data["High"] <= upper_bound) & (data["High"] >= lower_bound)]

In [ ]:
data.shape

####Column 'Low'

In [ ]:
data["Low"].describe()

In [ ]:
sns.boxplot(data = data, x = "Low")
plt.show()

In [ ]:
low_arr = np.array(data["Low"])

In [ ]:
Q1 = np.percentile(low_arr, 25)
Q3 = np.percentile(low_arr, 75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1: ", Q1)
print("Q3: ", Q3)
print("IQR: ", IQR)
print()
print("Lower bound: ", lower_bound)
print("Upper bound: ", upper_bound)

In [ ]:
data[(data["Low"] > upper_bound) | (data["Low"] < lower_bound)]

In [ ]:
data[(data["Low"] > upper_bound) | (data["Low"] < lower_bound)].shape

In [ ]:
data = data[(data["Low"] <= upper_bound) & (data["Low"] >= lower_bound)]

In [ ]:
data.shape

####Columns 'Close'

In [ ]:
data["Close"].describe()

In [ ]:
sns.boxplot(data = data, x = "Close")
plt.show()

In [ ]:
close_arr = np.array(data["Close"])

In [ ]:
Q1 = np.percentile(close_arr, 25)
Q3 = np.percentile(close_arr, 75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1: ", Q1)
print("Q3: ", Q3)
print("IQR: ", IQR)
print()
print("Lower bound: ", lower_bound)
print("Upper bound: ", upper_bound)

In [ ]:
data[(data["Close"] > upper_bound) | (data["Close"] < lower_bound)]

In [ ]:
data[(data["Close"] > upper_bound) | (data["Close"] < lower_bound)].shape

In [ ]:
data = data[(data["Close"] <= upper_bound) & (data["Close"] >= lower_bound)]

In [ ]:
data.shape

####Creating New Datasets for Differently Skewed Columns

In [ ]:
df_log_trans = pd.DataFrame()

df_cbrt_trans = pd.DataFrame()

In [ ]:
df_log_trans["Month_Sin"] = data["Month_Sin"]

df_log_trans["Month_Cos"] = data["Month_Cos"]

df_log_trans.head()

In [ ]:
df_cbrt_trans["Month_Sin"] = data["Month_Sin"]

df_cbrt_trans["Montth_Cos"] = data["Month_Cos"]

df_cbrt_trans.head()

####Handling Skewness --> 'Open' Column

df['Column'].skew() value interpretation

* < 0.5  --> Approximately symmetric

* 0.5 - 1 --> Moderately Skewed

* greater than 1 --> Highly Skewed

In [ ]:
sns.histplot(data = data, x = "Open", kde = True)
plt.show()

In [ ]:
# Highly Skewed
data["Open"].skew()

#####Log Transformation

In [ ]:
df_1 = np.log1p(data["Open"]).reset_index()

df_1.head()

In [ ]:
sns.histplot(data = df_1, x = "Open", kde = True)
plt.show()

In [ ]:
df_1["Open"].skew()

In [ ]:
df_log_trans["Open"] = df_1["Open"]

df_log_trans.head()

#####Cube Root Transformation

In [ ]:
df_2 = np.cbrt(data["Open"]).reset_index()

df_2.head()

In [ ]:
sns.histplot(data = df_2, x = "Open", kde = True)
plt.show()

In [ ]:
# Approximately symmetric
df_2["Open"].skew()

In [ ]:
df_cbrt_trans["Open"] = df_2["Open"]

df_cbrt_trans.head()

####Handling Skewness --> 'High' Column

In [ ]:
sns.histplot(data = data, x = "High", kde = True)
plt.show()

In [ ]:
data["High"].skew()

#####Log Transformation

In [ ]:
df_1 = np.log1p(data["High"]).reset_index()

df_1.head()

In [ ]:
sns.histplot(data = df_1, x = "High", kde = True)
plt.show()

In [ ]:
df_1["High"].skew()

In [ ]:
df_log_trans["High"] = df_1["High"]

df_log_trans.head()

#####Cube Root Transformation

In [ ]:
df_2 = np.cbrt(data["High"]).reset_index()

df_2.head()

In [ ]:
sns.histplot(data = df_2, x = "High", kde = True)
plt.show()

In [ ]:
df_2["High"].skew()

In [ ]:
df_cbrt_trans["High"] = df_2["High"]

df_cbrt_trans.head()

####Handling Skewness --> 'Low' Column

In [ ]:
data["Low"].skew()

In [ ]:
sns.histplot(data = data, x = "Low", kde = True)
plt.show()

#####Log Transformation

In [ ]:
np.log1p(data["Low"]).skew()

In [ ]:
df_1 = np.log1p(data["Low"]).reset_index()

df_1.head()

In [ ]:
sns.histplot(data = df_1, x = "Low", kde = True)
plt.show()

In [ ]:
df_log_trans["Low"] = df_1["Low"]

df_log_trans.head()

#####Cube Root Transformation

In [ ]:
np.cbrt(data["Low"]).skew()

In [ ]:
df_2 = np.cbrt(data["Low"]).reset_index()

df_2.head()

In [ ]:
sns.histplot(data = df_2, x = "Low", kde = True)
plt.show()

In [ ]:
df_cbrt_trans["Low"] = df_2["Low"]

df_cbrt_trans.head()

####Handling Skewness --> 'Close' Column

In [ ]:
data["Close"].skew()

In [ ]:
sns.histplot(data = data, x = "Close", kde = True)
plt.show()

#####Log Transformation

In [ ]:
np.log1p(data["Close"]).skew()

In [ ]:
df_1 = np.log1p(data["Close"]).reset_index()

df_1.head()

In [ ]:
sns.histplot(data = df_1, x = "Close", kde = True)
plt.show()

In [ ]:
df_log_trans["Close"] = df_1["Close"]

df_log_trans.head()

#####Cube Root Transformation

In [ ]:
np.cbrt(data["Close"]).skew()

In [ ]:
df_2 = np.cbrt(data["Close"]).reset_index()

df_2.head()

In [ ]:
sns.histplot(data = df_2, x = "Close", kde = True)
plt.show()

In [ ]:
df_cbrt_trans["Close"] = df_2["Close"]

df_cbrt_trans.head()

#####Dropping null values

In [ ]:
df_log_trans.isna().sum()

In [ ]:
df_log_trans = df_log_trans.dropna()

In [ ]:
df_cbrt_trans.isna().sum()

In [ ]:
df_cbrt_trans = df_cbrt_trans.dropna()

In [ ]:
df_log_trans.isna().sum()

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1 : Distribution Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize = (12, 8))

# Histogram 1:  Distribution of 'Open'
sns.histplot(data["Open"], kde = True, ax = axes[0, 0])
axes[0, 0].set_title("Distribution of 'Open'")

# Histogram 2:  Distribution of 'Open'
sns.histplot(data["High"], kde = True, ax = axes[0, 1])
axes[0, 1].set_title("Distribution of 'High'")

# Histogram 3:  Distribution of 'Low'
sns.histplot(data["Low"], kde = True, ax = axes[1, 0])
axes[1, 0].set_title("Distribution of 'Low'")

# Histogram 4:  Distribution of 'Close'
sns.histplot(data["Close"], kde = True, ax = axes[1, 1])
axes[1, 1].set_title("Distribution of 'Close'")

plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Histogram is used to visualize the distribution of numerical variable.

##### 2. What is/are the insight(s) found from the chart?

* All variables are right skewed.

* Most of the stock prices --> Mid Price

#### Chart - 2 : Line Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize = (12, 6))

# LinePlot 1
sns.lineplot(data = data, x = "Month", y = "Open", ax = axes[0, 0])
axes[0, 0].set_title("Month vs Open")

# LinePlot 2
sns.lineplot(data = data, x = "Month", y = "High", ax = axes[0, 1])
axes[0, 1].set_title("Month vs High")

# LinePlot 3
sns.lineplot(data = data, x = "Month", y = "Low", ax = axes[1, 0])
axes[1, 0].set_title("Month vs Low")

# LinePlot 4
sns.lineplot(data = data, x = "Month", y = "Close", ax = axes[1, 1])
axes[1, 1].set_title("Month vs Close")

plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

LinePlot : To show trends over time.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

#### Chart - 3 : Correlation Heatmap

In [ ]:
corr = data[["Open", "High", "Low", "Close"]].corr()

plt.figure(figsize = (6, 5))

sns.heatmap(corr, annot = True, cmap = "coolwarm")

plt.title("Correlation Matrix")
plt.show()

##### 1. Why did you pick the specific chart?

To visualize the correlation between variables.

##### 2. What is/are the insight(s) found from the chart?

All the four variable are highly correlated.

#### Chart - 4 : Pair Plot

In [ ]:
sns.pairplot(data[["Open", "High", "Low", "Close"]])

plt.show()

##### 1. Why did you pick the specific chart?

To check relationship between every pair of variables. Distributiona and correlation together.

##### 2. What is/are the insight(s) found from the chart?

Variable are highly correlated.

#### Chart - 5 : Scatter Plot - Open vs Close

In [ ]:
sns.scatterplot(data = data, x = "Open", y = "Close")

plt.title("Open vs Close")
plt.show()

##### 1. Why did you pick the specific chart?

Scatterplot: to visualize the relationship and outliers if present any.

##### 2. What is/are the insight(s) found from the chart?

Few Outliers are there. Highly correlated.

#### Chart - 6 : Monthly Average Price Trend

In [ ]:
monthly_avg = data.groupby("Month")["Close"].mean().reset_index()

sns.lineplot(data = monthly_avg, x = "Month", y = "Close", marker = "o")

plt.title("Monthly Average Price Trend")
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

## ***5. ML Model Implementation***

### ML Model - 1 : Linear Regression

####Log Transformation Dataset

In [ ]:
df_log_trans.head()

In [ ]:
X_log = df_log_trans[["Month_Sin", "Month_Cos", "Open", "High", "Low"]]

y_log = df_log_trans["Close"]

In [ ]:
model_1_log = LinearRegression()

In [ ]:
# Splitting dataset
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X_log, y_log, test_size=0.3, random_state=42)

In [ ]:
# Training model
model_1_log.fit(X_train_log, y_train_log)

In [ ]:
# Predictions
y_pred_log = model_1_log.predict(X_test_log)

In [ ]:
# Evaluating model
r2 = r2_score(y_test_log, y_pred_log)
mse = mean_squared_error(y_test_log, y_pred_log)
mae = mean_absolute_error(y_test_log, y_pred_log)

In [ ]:
# Printing Evaluation
print("R2: ", r2)
print("MSE: ", mse)
print("MAE: ", mae)

####Cube Root Transformation

In [ ]:
df_cbrt_trans.head()

In [ ]:
X_cbrt = df_cbrt_trans[["Month_Sin", "Montth_Cos", "Open", "High", "Low"]]

y_cbrt = df_cbrt_trans["Close"]

In [ ]:
model_1_cbrt = LinearRegression()

In [ ]:
# Splitting dataset
X_train_cbrt, X_test_cbrt, y_train_cbrt, y_test_cbrt = train_test_split(X_cbrt, y_cbrt, test_size=0.3, random_state=42)

In [ ]:
# Training model
model_1_cbrt.fit(X_train_cbrt, y_train_cbrt)

In [ ]:
# Predictions
y_pred_cbrt = model_1_cbrt.predict(X_test_cbrt)

In [ ]:
# Evaluating model
r2 = r2_score(y_test_cbrt, y_pred_cbrt)
mse = mean_squared_error(y_test_cbrt, y_pred_cbrt)
mae = mean_absolute_error(y_test_cbrt, y_pred_cbrt)

In [ ]:
# Printing Evaluation
print("R2: ", r2)
print("MSE: ", mse)
print("MAE: ", mae)

### ML Model - 2 : Random Forest Regressor

####Log Transformation Dataset

In [ ]:
model_2_log = RandomForestRegressor(n_estimators = 100, random_state = 42)

In [ ]:
model_2_log.fit(X_train_log, y_train_log)

In [ ]:
y_pred_log_model_2 = model_2_log.predict(X_test_log)

In [ ]:
# Evaluating model
r2 = r2_score(y_test_log, y_pred_log_model_2)
mse = mean_squared_error(y_test_log, y_pred_log_model_2)
mae = mean_absolute_error(y_test_log, y_pred_log_model_2)

In [ ]:
# Printing Evaluation
print("R2: ", r2)
print("MSE: ", mse)
print("MAE: ", mae)

####Cube Root Transformation Dataset

In [ ]:
model_2_cbrt = RandomForestRegressor(n_estimators = 100, random_state = 42)

In [ ]:
model_2_cbrt.fit(X_train_cbrt, y_train_cbrt)

In [ ]:
y_pred_cbrt_model_2 = model_2_cbrt.predict(X_test_cbrt)

In [ ]:
# Evaluating model
r2 = r2_score(y_test_cbrt, y_pred_cbrt_model_2)
mse = mean_squared_error(y_test_cbrt, y_pred_cbrt_model_2)
mae = mean_absolute_error(y_test_cbrt, y_pred_cbrt_model_2)

In [ ]:
# Printing Evaluation
print("R2: ", r2)
print("MSE: ", mse)
print("MAE: ", mae)

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

* Mean Absolute Error (MAE): Measures the average absolute difference between predicted and actual values. It is in the same units as the target and is robust to outliers.

* Mean Squared Error (MSE): MSE squares the errors before averaging, penalizing larger errors more.

* R square (Coefficient of Determination):  Represents the proportion of variance in the target explained by the model

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Linear Regression Model created using log transformed dataset, model_1_log. Because all the metrices are perfect in this model.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

####Using Pandas

* Positive coefficient --> increases prediction

* Negative coefficient --> decreases prediction

* Larger absolute value --> more influence

In [ ]:
X_log.columns

In [ ]:
print(model_1_log.coef_)

In [ ]:
feature_importance = pd.DataFrame({
    "Feature" : X_log.columns,
    "Coefficient" : model_1_log.coef_
})

feature_importance

####Using SHAP

In [ ]:
explainer = shap.Explainer(model_1_log, X_train_log)

In [ ]:
shap_values = explainer(X_test_log)

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
shap.summary_plot(shap_values, X_test_log)

# **Conclusion**

In this project, machine learning techniques were applied to predict the monthly closing stock price of Yes Bank using historical stock market data consisting of opening, highest, lowest, and closing prices. The project involved data preprocessing, exploratory data analysis, feature engineering, model building, evaluation, and model explainability.

Several analyses were performed to understand the behavior of Yes Bank's stock prices, particularly considering the impact of significant events such as the fraud case involving Rana Kapoor. Exploratory data analysis revealed strong relationships among the stock price variables, indicating that historical price movements contain valuable information for predicting future closing prices.

The final regression model achieved excellent performance with an R² score of 0.9934, indicating that approximately 99.34% of the variation in the closing stock price is explained by the model. The low Mean Squared Error (MSE) of 0.0033 and Mean Absolute Error (MAE) of 0.045 further demonstrate the model's high prediction accuracy and reliability.

Model explainability using SHAP revealed that the High price was the most influential feature in predicting the closing price, followed by the Low price and Open price. The cyclic month-based features (Month_Sin and Month_Cos) had minimal impact on the predictions, suggesting that stock price levels contributed much more to the model's decisions than seasonal monthly patterns.

Feature importance ranking obtained from SHAP analysis:

1. High Price (+0.46)
2. Low Price (+0.37)
3. Open Price (+0.26)
4. Month_Sin (+0.01)
5. Month_Cos (+0.00)

Overall, the project successfully demonstrates the effectiveness of machine learning for stock price prediction using historical market data. The results indicate that the developed model can accurately predict the monthly closing price of Yes Bank and provide meaningful insights into the factors influencing stock price movements. While external factors such as economic conditions, market sentiment, and unforeseen events can affect stock performance, the proposed approach offers a strong data-driven framework for financial forecasting and analysis.
